In [4]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import os

# --- SET YOUR PATH AND FILENAMES HERE ---
base_path = r"C:\Calculix\ccx_2.23_wsl\Stiffness matrices\Scenario2 Full K"
file_base_name = "cantilever_plusfifty_shell_k"  # The name before .sti, .mas, .dof

sti_file = os.path.join(base_path, file_base_name + ".sti")
mas_file = os.path.join(base_path, file_base_name + ".mas")
dof_file = os.path.join(base_path, file_base_name + ".dof")

def read_ccx_matrix(file_path):
    """Reads CalculiX .sti or .mas files into a CSR sparse matrix."""
    if not os.path.exists(file_path):
        print(f"Error: File not found: {file_path}")
        return None

    rows, cols, data = [], [], []
    max_idx = 0

    with open(file_path, 'r') as f:
        for line in f:
            parts = line.split()
            if len(parts) != 3: continue

            i = int(parts[0]) - 1
            j = int(parts[1]) - 1
            val = float(parts[2])

            rows.append(i)
            cols.append(j)
            data.append(val)
            max_idx = max(max_idx, i, j)

    n = max_idx + 1
    matrix = sp.coo_matrix((data, (rows, cols)), shape=(n, n))
    # Enforce symmetry (stripping the diagonal to avoid doubling it)
    matrix = matrix + matrix.T - sp.diags(matrix.diagonal())
    return matrix.tocsr()

def get_tip_dof(dof_path):
    """Optional: Reads .dof to find the highest node number (the tip) Z-direction."""
    # Assuming highest node number is the tip.
    # .dof file format: node_number.dof_type (1=x, 2=y, 3=z)
    try:
        with open(dof_path, 'r') as f:
            lines = f.readlines()
            # Return the last line's index (0-based) as a guess for the tip
            return len(lines) - 1
    except:
        return -1

# --- 1. Load the Matrices ---
print(f"Loading files from: {base_path}...")
K = read_ccx_matrix(sti_file)
M = read_ccx_matrix(mas_file)

if K is not None and M is not None:
    print(f"Success! Matrix Size: {K.shape[0]} x {K.shape[0]}")

    # --- 2. Solve Generalized Eigenvalues ---
    # Solving [K]{phi} = lambda [M]{phi}
    # This correctly handles the 'extra' shell DOFs and Lagrange multipliers
    try:
        # sigma=0 finds the lowest frequencies (fundamental modes)
        vals, vecs = spla.eigsh(K, M=M, k=6, sigma=0, which='LM')

        # vals are omega^2. Natural frequencies f = sqrt(vals)/(2*pi)
        freqs_hz = np.sqrt(np.abs(vals)) / (2 * np.pi)

        print("\n--- Generalized Modal Results (The Real Physics) ---")
        for i, f in enumerate(np.sort(freqs_hz)):
            print(f"Mode {i+1}: {f:.4f} Hz")
    except Exception as e:
        print(f"\nEigenvalue Error: {e}")
        print("Note: If M has zeros on the diagonal (Lagrange Multipliers), try a small sigma shift (e.g., sigma=1.0)")

    # --- 3. Static Compliance (The Reliable Metric) ---
    tip_index = get_tip_dof(dof_file)
    f_vec = np.zeros(K.shape[0])
    # Apply load to the identified tip or the very last DOF
    f_vec[tip_index if tip_index != -1 else -1] = 1.0

    try:
        u = spla.spsolve(K, f_vec)
        compliance = np.dot(f_vec, u)
        print(f"\n--- Static Compliance ---")
        print(f"Tip Displacement for Unit Load: {u[tip_index if tip_index != -1 else -1]:.6e}")
        print(f"Work Done (Invariant Metric): {compliance:.6e}")
    except Exception as e:
        print(f"Compliance solve failed: {e}")

else:
    print("Could not proceed without K and M matrices.")

Loading files from: C:\Calculix\ccx_2.23_wsl\Stiffness matrices\Scenario2 Full K...
Success! Matrix Size: 996 x 996

--- Generalized Modal Results (The Real Physics) ---
Mode 1: 21.1056 Hz
Mode 2: 31.4317 Hz
Mode 3: 131.2457 Hz
Mode 4: 192.3499 Hz
Mode 5: 356.1709 Hz
Mode 6: 363.3500 Hz

--- Static Compliance ---
Tip Displacement for Unit Load: 1.473862e-09
Work Done (Invariant Metric): 1.473862e-09
